# TVAE

Third of the four generation notebooks, following the same shape as the previous two.

TVAE takes a different neural approach from CTGAN. Rather than two networks competing, one
network compresses each record into a small internal summary and a second reconstructs
records from that summary. New records are generated by sampling fresh points in the
summary space and decoding them. There is no adversarial game, which generally makes it
more stable to train.

The trade-off, noted in the literature review, is that the compression step can smooth away
rare or unusual patterns, since those are the easiest thing to lose when records are
squeezed through a small summary. The fidelity checks on rare categories will show whether
that happened here.

## Setup

In [1]:
%pip install -q pandas pyarrow sdv

## Working folder

Sets the project folder so everything the pipeline writes, the cohort, the synthetic
datasets, the outputs and the figures, persists between sessions rather than sitting on
temporary storage.

The cohort and the synthetic datasets derive from MIMIC-IV, which is credentialed data
under a PhysioNet data use agreement. Keep the folder private, do not share it, and
delete the data once the work is finished.

In [2]:
import os
from pathlib import Path

# Use the shared project folder when one is available, otherwise stay in the
# current directory.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    project_dir = Path("/content/drive/MyDrive/mimic-synthetic-pipeline")
    project_dir.mkdir(parents=True, exist_ok=True)
    os.chdir(project_dir)
    print(f"Working folder: {project_dir}")
except ImportError:
    print("Using the local working folder.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working folder set to Google Drive: /content/drive/MyDrive/mimic-synthetic-pipeline


## Hardware check

TVAE is a neural network like CTGAN, so the same hardware note applies: minutes on a GPU,
potentially more than an hour on a CPU.

In [3]:
import torch

cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
if cuda_available:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print(
        "No GPU detected. TVAE will train on CPU, which can take over an hour on this "
        "cohort. If a GPU was expected, enable it in the session settings, then "
        "restart the session and re-run from the beginning."
    )

CUDA available: True
GPU: NVIDIA L4


## Training data

In [4]:
from pathlib import Path

import pandas as pd

if not Path("data/train.parquet").exists():
    raise FileNotFoundError(
        "data/train.parquet not found. Run the extraction and preparation steps first."
    )

TARGET = "readmitted_30d"
train_df = pd.read_parquet("data/train.parquet")
print(f"Training data: {len(train_df):,} admissions, readmission rate {train_df[TARGET].mean():.4f}")

Training data: 427,408 admissions, readmission rate 0.2067


## Hyperparameter search

An earlier round of this project ran every method at library defaults, with the training
budget set by the convergence rule. This section establishes whether TVAE performs better under other
settings, so the comparison between methods is not merely a comparison of defaults.

### Method

The search uses successive halving. Every configuration is screened on a small sample at a
reduced budget, and only the strongest few are promoted to a larger sample, the full budget
and repeated seeds. Configurations that look unpromising are therefore abandoned early,
which is where the saving in computation comes from, and the survivors are assessed with
enough repetition that the choice between them is not made on a single noisy run.

Neither library used in this project supports resuming training or validation-based early
stopping at a practical cost, so stopping is applied at the level of the configuration
rather than the epoch. The convergence rule used elsewhere in this project is applied to
each trial's loss curve and reported alongside its score, so a configuration that had not
finished training is visible rather than silently accepted.

Selection uses utility measured on the validation split. The test set is never involved, so
no part of the search can influence the reported results. Fidelity is recorded for every
trial but is not optimised, which means any movement in it is a consequence of selecting for
utility rather than a target of the search. That relationship is itself a finding worth
reporting.

Configurations whose mean scores fall within 0.005 of the best are reported as
indistinguishable, following the same reasoning applied to the differences between methods.
Where several are tied, the simplest should be preferred.

### Parameters varied

The size of the internal representation, the encoder and decoder widths, weight decay, batch size and `loss_factor`. `loss_factor` is the important one: it weights reconstruction against the divergence term and therefore governs how much detail survives compression. Rare-category smoothing is this method's documented weakness and race was its worst column by a wide margin, so a search omitting this parameter would not test the known failure mode.

### Cost and outputs

Twelve configurations screened, three promoted with three seeds each. Expect roughly one and a half to two hours.

Three files are written: every individual trial, a summary averaged over seeds, and the
selected configuration as JSON. The training cell below reads the selected
configuration automatically and applies it to the full training split. Nothing here overwrites the saved
synthetic datasets.

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import torch
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# --- Search budget. Reduce these if the search needs to finish sooner. ----------------
SCREEN_ROWS = 25_000     # rows per trial in the screening round
SCREEN_FRACTION = 0.33   # fraction of the full training budget used when screening
PROMOTE_ROWS = 100_000   # rows per trial once a configuration is promoted
PROMOTE_KEEP = 3         # configurations carried into the promotion round
PROMOTE_SEEDS = (0, 1, 2)  # repeats per promoted configuration
TIE_THRESHOLD = 0.005    # ROC AUC difference treated as indistinguishable
FIDELITY_TOLERANCE = 1.5 # a candidate may not worsen KS or TVD beyond this multiple of
                         # the library default, however much utility it gains

NUMERIC_COLS = ["age_at_admission", "length_of_stay_days"]
CATEGORICAL_COLS = [
    "gender", "admission_type", "admission_location", "insurance",
    "marital_status", "race", "language", "had_icu_stay",
]

for _required in ["data/train_fit.parquet", "data/train_val.parquet"]:
    if not Path(_required).exists():
        raise FileNotFoundError(
            f"{_required} not found. Re-run 02_data_preparation.ipynb, which writes the "
            "validation split this search depends on."
        )

fit_df = pd.read_parquet("data/train_fit.parquet")
val_df = pd.read_parquet("data/train_val.parquet")
for _c in fit_df.columns:
    if str(fit_df[_c].dtype) == "Int64":
        fit_df[_c] = fit_df[_c].astype("int64")
        val_df[_c] = val_df[_c].astype("int64")

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

print(f"Fitting split {len(fit_df):,} rows | validation split {len(val_df):,} rows")


def make_classifier() -> Pipeline:
    """The classifier from the main evaluation, so scores are directly comparable."""
    preprocess = ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS),
    ])
    return Pipeline([
        ("preprocess", preprocess),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])


def utility_on_validation(synthetic: pd.DataFrame) -> float:
    """Train on synthetic, score on the real validation split. The test set is never used."""
    clf = make_classifier()
    clf.fit(synthetic.drop(columns=[TARGET]), synthetic[TARGET])
    proba = clf.predict_proba(val_df.drop(columns=[TARGET]))[:, 1]
    return float(roc_auc_score(val_df[TARGET], proba))


def _tvd(real_col: pd.Series, syn_col: pd.Series) -> float:
    p = real_col.value_counts(normalize=True)
    q = syn_col.value_counts(normalize=True)
    return 0.5 * sum(abs(p.get(c, 0.0) - q.get(c, 0.0)) for c in p.index.union(q.index))


def fidelity_summary(synthetic: pd.DataFrame, real: pd.DataFrame) -> tuple:
    ks = float(np.mean([
        ks_2samp(real[c].astype(float), synthetic[c].astype(float)).statistic
        for c in NUMERIC_COLS
    ]))
    tvd = float(np.mean([
        _tvd(real[c].astype(str), synthetic[c].astype(str)) for c in CATEGORICAL_COLS
    ]))
    return ks, tvd


def converged(loss_series) -> tuple:
    """The stopping rule used elsewhere: mean loss over the final fifth of training
    against the fifth before it. Returns the percentage improvement and a flag."""
    if loss_series is None or len(loss_series) < 10:
        return float("nan"), None
    values = np.asarray(loss_series, dtype=float)
    n = len(values)
    previous = values[int(n * 0.6):int(n * 0.8)].mean()
    final = values[int(n * 0.8):].mean()
    improvement = (previous - final) / abs(previous) * 100
    return float(improvement), bool(improvement <= 1.0)


def set_seed(seed: int) -> None:
    """The sdv synthesizers expose no seed argument, so the global generators are set."""
    np.random.seed(seed)
    torch.manual_seed(seed)


def run_search(space, fit_and_sample, method_slug, full_budget):
    """Successive halving. Every configuration is screened on a small sample at a reduced
    budget; the best few are promoted to a larger sample, a full budget and repeated seeds.
    Unpromising configurations are therefore stopped early, which is where the compute
    saving comes from."""
    screen_sample = fit_df.sample(n=min(SCREEN_ROWS, len(fit_df)), random_state=0).reset_index(drop=True)
    screen_budget = max(1, int(full_budget * SCREEN_FRACTION))

    print(f"\n{'=' * 78}")
    print(f"ROUND 1, screening {len(space)} configurations")
    print(f"{len(screen_sample):,} rows, budget {screen_budget}, one seed each")
    print(f"{'=' * 78}", flush=True)

    screened = []
    for i, (label, config) in enumerate(space, start=1):
        print(f"\n[{i}/{len(space)}] {label}", flush=True)
        t0 = time.time()
        try:
            set_seed(0)
            synthetic, losses = fit_and_sample(config, screen_sample, screen_budget)
            auc = utility_on_validation(synthetic)
            ks, tvd = fidelity_summary(synthetic, screen_sample)
            improvement, is_converged = converged(losses)
            seconds = time.time() - t0
            screened.append({
                "config_label": label, "config": json.dumps(config), "round": "screen",
                "val_roc_auc": auc, "mean_ks": ks, "mean_tvd": tvd,
                "loss_improvement_pct": improvement, "converged": is_converged,
                "seconds": seconds,
            })
            flag = "" if is_converged is None else ("converged" if is_converged else "NOT converged")
            print(f"      ROC AUC {auc:.4f} | KS {ks:.4f} | TVD {tvd:.4f} | {seconds:.0f}s {flag}", flush=True)
        except Exception as exc:  # noqa: BLE001
            print(f"      failed: {type(exc).__name__}: {exc}", flush=True)

    if not screened:
        raise RuntimeError("Every configuration failed during screening.")

    screen_df = pd.DataFrame(screened).sort_values("val_roc_auc", ascending=False)

    # Utility alone is not a sufficient selection criterion. A configuration can raise the
    # downstream score while badly degrading the distributions, which would be a poor
    # outcome for a project whose argument is that these dimensions must be read together.
    # Candidates are therefore restricted to those whose fidelity is no worse than the
    # first configuration in the search space, the library default, by more than the
    # tolerance below. Utility decides the ranking within that set.
    baseline_label = space[0][0]
    baseline = screen_df[screen_df["config_label"] == baseline_label]
    eligible = screen_df
    if not baseline.empty:
        base_ks = float(baseline.iloc[0]["mean_ks"])
        base_tvd = float(baseline.iloc[0]["mean_tvd"])
        limit_ks = base_ks * FIDELITY_TOLERANCE + 1e-6
        limit_tvd = base_tvd * FIDELITY_TOLERANCE + 1e-6
        eligible = screen_df[(screen_df["mean_ks"] <= limit_ks)
                             & (screen_df["mean_tvd"] <= limit_tvd)]
        excluded = screen_df[~screen_df["config_label"].isin(eligible["config_label"])]
        print()
        print(f"Fidelity guardrail: KS <= {limit_ks:.4f}, TVD <= {limit_tvd:.4f}")
        print(f"  ({FIDELITY_TOLERANCE}x the baseline configuration '{baseline_label}')")
        if len(excluded):
            print(f"  Excluded {len(excluded)} configuration(s) that raised utility at the "
                  f"cost of fidelity:")
            for _, r in excluded.iterrows():
                print(f"    {r['config_label']:<42} ROC AUC {r['val_roc_auc']:.4f}  "
                      f"KS {r['mean_ks']:.4f}  TVD {r['mean_tvd']:.4f}")
        if eligible.empty:
            print("  No configuration met the guardrail, so it has been relaxed for this run.")
            eligible = screen_df

    promoted = eligible.head(PROMOTE_KEEP)

    print(f"\n{'=' * 78}")
    print(f"ROUND 2, promoting the top {len(promoted)} of {len(screen_df)}")
    print(f"{min(PROMOTE_ROWS, len(fit_df)):,} rows, budget {full_budget}, "
          f"{len(PROMOTE_SEEDS)} seeds each")
    print(f"{'=' * 78}", flush=True)

    promote_sample = fit_df.sample(n=min(PROMOTE_ROWS, len(fit_df)), random_state=0).reset_index(drop=True)
    promoted_rows = []
    for i, row in enumerate(promoted.itertuples(index=False), start=1):
        config = json.loads(row.config)
        print(f"\n[{i}/{len(promoted)}] {row.config_label}", flush=True)
        for seed in PROMOTE_SEEDS:
            t0 = time.time()
            try:
                set_seed(seed)
                synthetic, losses = fit_and_sample(config, promote_sample, full_budget, seed=seed)
                auc = utility_on_validation(synthetic)
                ks, tvd = fidelity_summary(synthetic, promote_sample)
                improvement, is_converged = converged(losses)
                promoted_rows.append({
                    "config_label": row.config_label, "config": row.config, "round": "promote",
                    "seed": seed, "val_roc_auc": auc, "mean_ks": ks, "mean_tvd": tvd,
                    "loss_improvement_pct": improvement, "converged": is_converged,
                    "seconds": time.time() - t0,
                })
                print(f"      seed {seed}: ROC AUC {auc:.4f} | KS {ks:.4f} | TVD {tvd:.4f}", flush=True)
            except Exception as exc:  # noqa: BLE001
                print(f"      seed {seed} failed: {type(exc).__name__}: {exc}", flush=True)

    promote_df = pd.DataFrame(promoted_rows)
    summary = (
        promote_df.groupby(["config_label", "config"])
        .agg(mean_roc_auc=("val_roc_auc", "mean"), sd_roc_auc=("val_roc_auc", "std"),
             mean_ks=("mean_ks", "mean"), mean_tvd=("mean_tvd", "mean"),
             runs=("val_roc_auc", "size"))
        .reset_index().sort_values("mean_roc_auc", ascending=False)
    )

    pd.concat([screen_df, promote_df], ignore_index=True).to_csv(
        out_dir / f"tuning_{method_slug}_trials.csv", index=False)
    summary.to_csv(out_dir / f"tuning_{method_slug}_summary.csv", index=False)

    print(f"\n{'=' * 78}")
    print("PROMOTION RESULTS, mean over seeds")
    print(f"{'=' * 78}")
    for _, r in summary.iterrows():
        sd = 0.0 if pd.isna(r["sd_roc_auc"]) else r["sd_roc_auc"]
        print(f"  {r['config_label']:<46} {r['mean_roc_auc']:.4f} +/- {sd:.4f}  "
              f"KS {r['mean_ks']:.4f}  TVD {r['mean_tvd']:.4f}  (n={r['runs']})")

    best = summary.iloc[0]
    tied = summary[summary["mean_roc_auc"] >= best["mean_roc_auc"] - TIE_THRESHOLD]

    checkpoint = {
        "method": method_slug,
        "selected_config": json.loads(best["config"]),
        "selected_label": best["config_label"],
        "mean_val_roc_auc": float(best["mean_roc_auc"]),
        "sd_val_roc_auc": None if pd.isna(best["sd_roc_auc"]) else float(best["sd_roc_auc"]),
        "seeds": list(PROMOTE_SEEDS),
        "screen_rows": int(min(SCREEN_ROWS, len(fit_df))),
        "promote_rows": int(min(PROMOTE_ROWS, len(fit_df))),
        "full_budget": full_budget,
        "tied_within_threshold": tied["config_label"].tolist(),
    }
    with open(out_dir / f"tuning_{method_slug}_selected.json", "w", encoding="utf-8") as fh:
        json.dump(checkpoint, fh, indent=2)

    print(f"\nSelected: {best['config_label']}")
    if len(tied) > 1:
        print(f"Within {TIE_THRESHOLD} of the best, so not separable on this evidence:")
        for label in tied["config_label"]:
            print(f"    {label}")
        print("Prefer the simplest of these, and say so in the write-up.")
    print(f"\nSaved outputs/tuning_{method_slug}_selected.json for the final run.")
    return summary


METHOD_SLUG = "tvae"
FULL_BUDGET = 600  # epochs, matching the converged budget used for the reported run

from sdv.metadata import Metadata
from sdv.single_table import TVAESynthesizer

# loss_factor is the important one. It weights reconstruction against the divergence term,
# so it governs directly how much detail survives compression. Rare-category smoothing is
# this method's documented weakness and its worst column by a wide margin was race, so a
# search that omitted this parameter would not be testing the known failure mode.
SEARCH_SPACE = [
    ("library defaults", {}),
    ("loss_factor=1", {"loss_factor": 1}),
    ("loss_factor=5", {"loss_factor": 5}),
    ("loss_factor=10", {"loss_factor": 10}),
    ("embedding_dim=256", {"embedding_dim": 256}),
    ("embedding_dim=64", {"embedding_dim": 64}),
    ("wide 256", {"compress_dims": (256, 256), "decompress_dims": (256, 256)}),
    ("narrow 64", {"compress_dims": (64, 64), "decompress_dims": (64, 64)}),
    ("l2scale=1e-4", {"l2scale": 1e-4}),
    ("l2scale=1e-6", {"l2scale": 1e-6}),
    ("batch_size=1000", {"batch_size": 1000}),
    ("loss_factor=5 + wide 256", {"loss_factor": 5, "compress_dims": (256, 256),
                                  "decompress_dims": (256, 256)}),
]


def fit_and_sample(config, data, budget, seed=0):
    metadata = Metadata.detect_from_dataframe(data, table_name="cohort")
    model = TVAESynthesizer(metadata, epochs=budget, verbose=False, **config)
    model.fit(data)
    synthetic = model.sample(num_rows=len(data))
    losses = None
    try:
        loss_df = model.get_loss_values()
        losses = loss_df.groupby("Epoch")["Loss"].mean().to_numpy()
    except Exception:  # noqa: BLE001
        pass
    return synthetic, losses

summary = run_search(SEARCH_SPACE, fit_and_sample, METHOD_SLUG, FULL_BUDGET)
summary


## Fitting and generating

Each method is trained to its own convergence rather than to a shared epoch count.
Matching epoch counts sounds fairer but is not, because an epoch means something different
for an adversarial game, a variational autoencoder and a diffusion model. Quality is
compared at each method's own ceiling, and the cost of reaching it is reported separately
in the timing log.

Unlike CTGAN, TVAE optimises a single reconstruction objective, so its loss curve is a
genuine convergence signal and the stopping rule can be applied properly. The budget is
set generously and the convergence check below reports whether the loss had flattened by
the end.

In [5]:
import time

from sdv.metadata import Metadata
from sdv.single_table import TVAESynthesizer

EPOCHS = 600  # Raised from 300, where the convergence check reported the loss still
              # improving by 1.95% across the final fifth, above the 1% threshold. At
              # roughly 7.3 seconds per epoch on an L4 this is about 73 minutes. If the
              # check still reports more than 1% at 600, raise it again and re-run.

import json as _json

# Set to False to reproduce the library-default run rather than the tuned one.
USE_TUNED_CONFIG = True

TUNED = {}
CONFIG_LABEL = "library defaults"
_selected = Path("outputs/tuning_tvae_selected.json")
if USE_TUNED_CONFIG and _selected.exists():
    with open(_selected, encoding="utf-8") as _fh:
        _payload = _json.load(_fh)
    TUNED = _payload["selected_config"]
    CONFIG_LABEL = _payload["selected_label"]
    print(f"Using the configuration chosen by the search: {CONFIG_LABEL}")
    print(f"  {TUNED}")
elif USE_TUNED_CONFIG:
    print("No search result found, so the library defaults are used.")
    print(f"  Run the search section at the end of this notebook to produce {_selected}.")
else:
    print("USE_TUNED_CONFIG is False, so the library defaults are used.")

metadata = Metadata.detect_from_dataframe(train_df, table_name="cohort")
model = TVAESynthesizer(metadata, epochs=EPOCHS, verbose=True, **TUNED)

t0 = time.time()
model.fit(train_df)
train_seconds = time.time() - t0

t0 = time.time()
synthetic_df = model.sample(num_rows=len(train_df))
generate_seconds = time.time() - t0

synthetic_df.to_parquet("data/synthetic_tvae.parquet", index=False)
print(f"Training took {train_seconds:.1f}s, generation took {generate_seconds:.1f}s")
print(f"Saved {len(synthetic_df):,} synthetic admissions to data/synthetic_tvae.parquet")

/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Loss: +03.38: 100%|██████████| 600/600 [1:08:41<00:00,  6.87s/it]


Training took 4223.3s, generation took 3.1s
Saved 427,408 synthetic admissions to data/synthetic_tvae.parquet


## Training convergence

TVAE optimises a single reconstruction objective with no adversarial component, so unlike
CTGAN its loss curve can be read as a convergence signal in the ordinary way. If the loss
is still falling appreciably at the last epoch, the model is undertrained.

The library records a loss value per batch, so values are averaged within each epoch to
give one point per epoch. Rather than eyeballing the shape, the cell compares the mean loss
over the final fifth of training against the fifth before it and prints the percentage
improvement, so the epoch count can be justified with a number.

In [ ]:
import csv

import matplotlib.pyplot as plt


def save_csv(df, path):
    """Write a dataframe to CSV, falling back to the standard library.

    Installing sdv can leave the session with a pandas whose CSV writer is broken
    ("AttributeError: 'Index' object has no attribute '_format_native_types'"),
    because pip replaces pandas on disk while the kernel still holds parts of the
    previous version. The stdlib writer does not touch pandas internals, so it works
    regardless. Reported either way so it is clear which path was taken.
    """
    try:
        df.to_csv(path, index=False)
        print(f"Saved {path}")
        return True
    except Exception as exc:  # noqa: BLE001
        try:
            with open(path, "w", newline="", encoding="utf-8") as fh:
                writer = csv.writer(fh)
                writer.writerow([str(c) for c in df.columns])
                writer.writerows(df.itertuples(index=False, name=None))
            print(f"Saved {path} (via the stdlib fallback; pandas raised {type(exc).__name__})")
            return True
        except Exception as exc2:  # noqa: BLE001
            print(f"Could not write {path}: {type(exc2).__name__}: {exc2}")
            return False


fig_dir = Path("figures")
fig_dir.mkdir(exist_ok=True)
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

loss_df = model.get_loss_values()  # one row per batch
per_epoch = loss_df.groupby("Epoch")["Loss"].mean()

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(per_epoch.index, per_epoch.values, color="#55A868")
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean loss")
ax.set_title(f"TVAE training convergence over {EPOCHS} epochs")
fig.tight_layout()
fig.savefig(fig_dir / "training_convergence_tvae.png", dpi=150)
plt.show()

# The convergence verdict is computed and printed BEFORE anything is written to disk,
# so a file-writing problem cannot cost the diagnostic after an hour of training.
n = len(per_epoch)
prev_fifth = per_epoch.iloc[int(n * 0.6):int(n * 0.8)].mean()
last_fifth = per_epoch.iloc[int(n * 0.8):].mean()
improvement = (prev_fifth - last_fifth) / abs(prev_fifth) * 100

print(f"Mean loss, epochs {int(n * 0.6) + 1}-{int(n * 0.8)}: {prev_fifth:.4f}")
print(f"Mean loss, final {n - int(n * 0.8)} epochs: {last_fifth:.4f}")
print(f"Improvement across the final fifth: {improvement:+.2f}%")

if improvement > 1.0:
    print(
        f"\nStill improving by more than 1% across the final fifth of training at "
        f"EPOCHS={EPOCHS}. Note the 1% figure is a stopping heuristic, not a law: weigh "
        f"it against the absolute change in the loss and the training time before "
        f"raising the budget again."
    )
else:
    print(
        f"\nFlat to within 1% across the final fifth, which is consistent with the "
        f"model having converged at EPOCHS={EPOCHS}."
    )

print()
save_csv(loss_df, out_dir / "loss_history_tvae.csv")

## Sanity checks

The same quick aggregate checks as the other generation notebooks.

In [11]:
checks = pd.DataFrame({
    "statistic": ["Readmission rate", "Mean age", "Mean length of stay (days)"],
    "real_training_data": [
        round(train_df[TARGET].mean(), 4),
        round(train_df["age_at_admission"].astype(float).mean(), 1),
        round(train_df["length_of_stay_days"].mean(), 2),
    ],
    "synthetic_data": [
        round(synthetic_df[TARGET].mean(), 4),
        round(synthetic_df["age_at_admission"].astype(float).mean(), 1),
        round(synthetic_df["length_of_stay_days"].mean(), 2),
    ],
})
checks

/usr/local/lib/python3.12/dist-packages/google/colab/_interactive_table_hint_button.py:178: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  df_html=dataframe._repr_html_(),  # pylint: disable=protected-access
/usr/local/lib/python3.12/dist-packages/google/colab/_interactive_table_hint_button.py:178: FutureWarning: RangeIndex.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  df_html=dataframe._repr_html_(),  # pylint: disable=protected-access


,statistic,real_training_data,synthetic_data
0,Readmission rate,0.2067,0.2026
1,Mean age,58.8000,59.1000
2,Mean length of stay (days),4.6400,4.4500


In [ ]:
# Record the hardware alongside the timings. Without it the cost comparison in the
# results chapter rests on recollection of which session used which accelerator.
try:
    import torch as _torch
    GPU_NAME = _torch.cuda.get_device_name(0) if _torch.cuda.is_available() else "CPU"
except Exception:  # noqa: BLE001
    GPU_NAME = "unknown"

# Append this run's timings to the shared generation log used by all four methods.
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
log_path = out_dir / "generation_log.csv"

entry = pd.DataFrame([{
    "method": "TVAE",
    "rows_generated": len(synthetic_df),
    "train_seconds": round(train_seconds, 1),
    "generate_seconds": round(generate_seconds, 1),
    "gpu": GPU_NAME,
    "config": CONFIG_LABEL,
}])
if log_path.exists():
    log = pd.read_csv(log_path)
    # Keyed on method and configuration, so the default and tuned runs both survive.
    if "config" in log.columns:
        log = log[~((log["method"] == "TVAE") & (log["config"] == CONFIG_LABEL))]
    else:
        log = log[log["method"] != "TVAE"]
    log = pd.concat([log, entry], ignore_index=True)
else:
    log = entry

# save_csv is defined in the convergence cell above and falls back to the standard
# library, because installing sdv can leave pandas' CSV writer broken for the rest of
# the session. This log is a result rather than a convenience, so it must not be lost
# to that. Run the convergence cell first if save_csv is not yet defined.
save_csv(log, log_path)
log